# End-to-end system identification: a symbolic ODE model

This notebook identifies the parameters of an epidemic model written as a system of
differential equations with SymPy, via `gsua_csb`'s `SymbolicODEModel`, and asks the same
question its user-defined companion asks: once the model fits, can the parameters actually be
recovered?

Here the answer depends on **when you stopped collecting data**, which makes it a concrete
illustration of why identifiability is a property of the experiment and not only of the model.

The model is the classical SIR system for a closed population of $N=1000$:

$$\frac{dS}{dt}=-\beta\frac{SI}{N},\qquad
  \frac{dI}{dt}=\beta\frac{SI}{N}-\gamma I,\qquad
  \frac{dR}{dt}=\gamma I$$

with two factors to identify: $\beta$ (transmission rate, 1/day) and $\gamma$ (recovery rate,
1/day). Their ratio is the basic reproduction number $R_0=\beta/\gamma$, the quantity an
epidemiologist actually wants.

*A MATLAB Live Script version of this example is available alongside it.*

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from gsua_csb import (SymbolicODEModel, UserFunctionModel, parameter_estimation,
                      identifiability_analysis, profile_likelihood)

N = 1000.0
t = sp.Symbol("t")
S, I, R = sp.symbols("S I R")
beta, gamma = sp.symbols("beta gamma")

odes = [-beta * S * I / N,
         beta * S * I / N - gamma * I,
         gamma * I]

## 1. Building the symbolic model

The factor vector is the initial conditions of the state variables first, in the order given by
`state_vars`, followed by the model parameters. Giving a factor a degenerate range fixes it:
here $S_0=999$, $I_0=1$ and $R_0=0$ are known, leaving $\beta$ and $\gamma$ to estimate.

Only $I(t)$ is measured, so the symbolic model is wrapped in a thin `UserFunctionModel` that
selects that one state as the observed output.

In [ ]:
TRUTH = np.array([999.0, 1.0, 0.0, 0.35, 0.10])       # S0, I0, R0, beta, gamma
BOUNDS = np.array([[999, 999], [1, 1], [0, 0], [0.15, 0.9], [0.02, 0.4]])
NAMES = ["S0", "I0", "R0i", "beta", "gamma"]

def build(domain):
    core = SymbolicODEModel(odes, [S, I, R], t, [beta, gamma], domain=domain,
                            names=NAMES, range=BOUNDS, nominal=TRUTH)
    observed = UserFunctionModel(
        func=lambda p, xd: core.evaluate(p, xd)[[1]],     # I(t) only
        names=NAMES, range=BOUNDS, nominal=TRUTH,
        domain=domain, output_names=["I"],
    )
    return core, observed

FREE = [3, 4]
print("free factors:", [NAMES[i] for i in FREE], "| true R0 =", TRUTH[3] / TRUTH[4])

## 2. Synthetic surveillance data

The true epidemic uses $\beta=0.35$ and $\gamma=0.10$, so $R_0=3.5$. Case counts are noisy in
a way that scales with their size, so the measurements carry Poisson-like noise.

In [ ]:
XFULL = np.linspace(1, 80, 20)
core_full, model_full = build(XFULL)

rng = np.random.default_rng(3)
clean_full = core_full.evaluate(TRUTH, XFULL)[[1]]
yfull = clean_full + np.sqrt(np.maximum(clean_full, 1)) * rng.standard_normal(clean_full.shape)

tdense = np.linspace(1, 80, 300)
plt.figure(figsize=(7, 4))
plt.plot(tdense, core_full.evaluate(TRUTH, tdense)[1], lw=1.8, label="true epidemic")
plt.plot(XFULL, yfull[0], "ko", ms=5, label="surveillance data")
plt.axvline(25, ls="--", c="grey"); plt.text(25.6, 300, "end of early phase", fontsize=9)
plt.xlabel("time (days)"); plt.ylabel("infected individuals I(t)")
plt.title("Simulated epidemic, $R_0$ = 3.5")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 3. Identifying from the full epidemic

With the whole curve available — growth, peak and decline — both factors are estimated by
multistart least squares.

In [ ]:
pe_full = parameter_estimation(model_full, XFULL, yfull, n=20,
                               solver="least_squares", margin=0.1, seed=0)
best_full = pe_full.x[np.argmin(pe_full.cost)]
ia_full = identifiability_analysis(model_full, pe_full.x, cost=pe_full.cost,
                                   cost_rtol=0.1, seed=0)
pl_full = profile_likelihood(model_full, XFULL, yfull, alpha=0.95, margin=0.1, params=FREE)

pd.DataFrame({
    "true":      TRUTH[FREE],
    "estimated": best_full[FREE].round(4),
    "CI_low":    pl_full.range[FREE, 0].round(4),
    "CI_high":   pl_full.range[FREE, 1].round(4),
    "width":     (pl_full.range[FREE, 1] - pl_full.range[FREE, 0]).round(4),
}, index=["beta", "gamma"])

In [ ]:
print(f"cost              : {pe_full.cost.min():.5g}")
print(f"R0 estimated      : {best_full[3] / best_full[4]:.3f}   (true 3.500)")

## 4. Identifying from the early phase only

Now suppose the analysis had to be done *during* the outbreak, with only the first 25 days in
hand — the situation every real-time epidemic assessment faces. Nothing about the model
changes; only the data window shrinks.

In [ ]:
XEARLY = np.linspace(1, 25, 12)
core_early, model_early = build(XEARLY)

rng = np.random.default_rng(3)
clean_early = core_early.evaluate(TRUTH, XEARLY)[[1]]
yearly = clean_early + np.sqrt(np.maximum(clean_early, 1)) * rng.standard_normal(clean_early.shape)

pe_early = parameter_estimation(model_early, XEARLY, yearly, n=20,
                                solver="least_squares", margin=0.1, seed=0)
best_early = pe_early.x[np.argmin(pe_early.cost)]
ia_early = identifiability_analysis(model_early, pe_early.x, cost=pe_early.cost,
                                    cost_rtol=0.1, seed=0)
pl_early = profile_likelihood(model_early, XEARLY, yearly, alpha=0.95, margin=0.1, params=FREE)

pd.DataFrame({
    "true":      TRUTH[FREE],
    "estimated": best_early[FREE].round(4),
    "CI_low":    pl_early.range[FREE, 0].round(4),
    "CI_high":   pl_early.range[FREE, 1].round(4),
    "width":     (pl_early.range[FREE, 1] - pl_early.range[FREE, 0]).round(4),
}, index=["beta", "gamma"])

In [ ]:
summary = pd.DataFrame({
    "full epidemic": [pe_full.cost.min(), ia_full.correlation[3, 4],
                      pl_full.range[3, 1] - pl_full.range[3, 0],
                      pl_full.range[4, 1] - pl_full.range[4, 0],
                      best_full[3] / best_full[4]],
    "early phase":   [pe_early.cost.min(), ia_early.correlation[3, 4],
                      pl_early.range[3, 1] - pl_early.range[3, 0],
                      pl_early.range[4, 1] - pl_early.range[4, 0],
                      best_early[3] / best_early[4]],
}, index=["best cost", "corr(beta, gamma)", "CI width beta", "CI width gamma", "R0 (true 3.5)"])
summary.round(4)

The estimates are still plausible, and the fit to the early data is *better* than the
full-epidemic fit was — fewer points, all of them on a smooth exponential rise. But the
intervals have widened and the correlation between $\beta$ and $\gamma$ has risen sharply.

During exponential growth the data constrain only the growth rate, roughly $\beta-\gamma$, so
any pair with the right difference reproduces the observations equally well. It takes the peak
— where susceptibles are depleted and the curve turns over — to separate them.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

x = np.arange(2); w = 0.35
ax[0].bar(x - w/2, [pl_full.range[i, 1] - pl_full.range[i, 0] for i in FREE], w, label="full epidemic")
ax[0].bar(x + w/2, [pl_early.range[i, 1] - pl_early.range[i, 0] for i in FREE], w, label="early phase")
ax[0].set_xticks(x); ax[0].set_xticklabels([r"$\beta$", r"$\gamma$"])
ax[0].set_ylabel("95% CI width"); ax[0].set_title("Confidence interval width")
ax[0].legend(); ax[0].grid(alpha=.3, axis="y")

ax[1].plot(tdense, core_full.evaluate(best_full, tdense)[1], lw=1.8, label="fit to full epidemic")
ax[1].plot(tdense, core_full.evaluate(best_early, tdense)[1], "--", lw=1.8,
           label="fit to early phase, extrapolated")
ax[1].plot(XFULL, yfull[0], "ko", ms=4, label="data")
ax[1].set_xlabel("time (days)"); ax[1].set_ylabel("I(t)")
ax[1].set_title("Where the early-phase fit leads")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. What the example shows

The same model and the same estimator produced two very different states of knowledge, and the
difference was the observation window rather than anything about the algorithm. The
better-fitting dataset was the less informative one.

Fit quality measures agreement with the points you have; identifiability measures whether those
points could have distinguished your parameters from the alternatives. They are different
questions, and only the second tells you whether an estimate is worth reporting.

For a real outbreak the consequence follows directly: an $R_0$ estimated before the peak
carries a confidence interval wide enough to change policy conclusions, and quoting the point
estimate alone would hide that.

Where a multistart run does spread across parameter space rather than converging to a point,
`identifiability_analysis` adds correlation structure and detection of multiple global minima,
`noise_floor` calibrates which fits to accept against the observation noise, and
`design_matrix(..., method="joint")` propagates the accepted set without destroying its
correlation structure.